# Assignment 4

In [2]:
import networkx as nx
import pandas as pd
import numpy as np
import pickle

---

## Part 1 - Random Graph Identification

For the first part of this assignment you will analyze randomly generated graphs and determine which algorithm created them.

In [4]:
G1 = nx.read_gpickle("assets/A4_P1_G1")
G2 = nx.read_gpickle("assets/A4_P1_G2")
G3 = nx.read_gpickle("assets/A4_P1_G3")
G4 = nx.read_gpickle("assets/A4_P1_G4")
G5 = nx.read_gpickle("assets/A4_P1_G5")
P1_Graphs = [G1, G2, G3, G4, G5]

<br>
`P1_Graphs` is a list containing 5 networkx graphs. Each of these graphs were generated by one of three possible algorithms:
* Preferential Attachment (`'PA'`)
* Small World with low probability of rewiring (`'SW_L'`)
* Small World with high probability of rewiring (`'SW_H'`)

Anaylze each of the 5 graphs using any methodology and determine which of the three algorithms generated each graph.

*The `graph_identification` function should return a list of length 5 where each element in the list is either `'PA'`, `'SW_L'`, or `'SW_H'`.*

In [22]:
def graph_identification():
    """
    Identifica el tipo de cada uno de los cinco grafos cargados desde archivos locales.
    
    Retorna:
        list: Una lista de strings indicando el tipo de cada grafo:
              - "tree" para árboles,
              - "scale_free" para grafos con alto clustering (posible modelo de Barabási–Albert),
              - "random" para grafos sin estructura específica clara.
    """
    import networkx as nx

    # Rutas de los archivos .gpickle
    paths = [f"assets/A4_P1_G{i}" for i in range(1, 6)]
    graphs = [nx.read_gpickle(p) for p in paths]
    result = []

    for G in graphs:
        if nx.is_tree(G):
            result.append("PA")
        elif nx.is_connected(G) and nx.average_clustering(G) > 0.45:
            result.append("SW_L")
        else:
            result.append("SW_H")

    return result

graph_identification()

['PA', 'SW_L', 'SW_L', 'PA', 'SW_H']

In [21]:
ans_one = graph_identification()
assert type(ans_one) == list, "You must return a list"


---

## Part 2 - Company Emails

For the second part of this assignment you will be working with a company's email network where each node corresponds to a person at the company, and each edge indicates that at least one email has been sent between two people.

The network also contains the node attributes `Department` and `ManagmentSalary`.

`Department` indicates the department in the company which the person belongs to, and `ManagmentSalary` indicates whether that person is receiving a managment position salary.

In [9]:
G = pickle.load(open('assets/email_prediction_NEW.txt', 'rb'))

print(f"Graph with {len(nx.nodes(G))} nodes and {len(nx.edges(G))} edges")

Graph with 1005 nodes and 16706 edges


### Part 2A - Salary Prediction

Using network `G`, identify the people in the network with missing values for the node attribute `ManagementSalary` and predict whether or not these individuals are receiving a managment position salary.

To accomplish this, you will need to create a matrix of node features of your choice using networkx, train a sklearn classifier on nodes that have `ManagementSalary` data, and predict a probability of the node receiving a managment salary for nodes where `ManagementSalary` is missing.



Your predictions will need to be given as the probability that the corresponding employee is receiving a managment position salary.

The evaluation metric for this assignment is the Area Under the ROC Curve (AUC).

Your grade will be based on the AUC score computed for your classifier. A model which with an AUC of 0.75 or higher will recieve full points.

Using your trained classifier, return a Pandas series of length 252 with the data being the probability of receiving managment salary, and the index being the node id.

    Example:
    
        1       1.0
        2       0.0
        5       0.8
        8       1.0
            ...
        996     0.7
        1000    0.5
        1001    0.0
        Length: 252, dtype: float64

In [10]:
list(G.nodes(data=True))[:5] # print the first 5 nodes

[(0, {'Department': 1, 'ManagementSalary': 0.0}),
 (1, {'Department': 1, 'ManagementSalary': nan}),
 (581, {'Department': 3, 'ManagementSalary': 0.0}),
 (6, {'Department': 25, 'ManagementSalary': 1.0}),
 (65, {'Department': 4, 'ManagementSalary': nan})]

In [12]:
def salary_predictions():
    import networkx as nx
    import pandas as pd
    import numpy as np
    import pickle
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.preprocessing import StandardScaler

    # Cargar el grafo
    G = pickle.load(open('assets/email_prediction_NEW.txt', 'rb'))

    # Crear DataFrame de nodos y atributos
    nodes = pd.DataFrame.from_dict(dict(G.nodes(data=True)), orient='index')

    # Features de red: grado, grado ponderado, clustering, pagerank
    nodes['degree'] = pd.Series(dict(G.degree()))
    nodes['degree_centrality'] = pd.Series(nx.degree_centrality(G))
    nodes['clustering'] = pd.Series(nx.clustering(G))
    nodes['pagerank'] = pd.Series(nx.pagerank(G))

    # Selección de features
    features = ['Department', 'degree', 'degree_centrality', 'clustering', 'pagerank']

    # Convertir Department a dummies
    dummies = pd.get_dummies(nodes['Department'], prefix='dept')
    X = pd.concat([nodes[features].drop('Department', axis=1), dummies], axis=1)

    # Separar nodos con y sin etiqueta
    X_train = X[nodes['ManagementSalary'].notnull()]
    y_train = nodes.loc[nodes['ManagementSalary'].notnull(), 'ManagementSalary']

    X_test = X[nodes['ManagementSalary'].isnull()]

    # Escalar
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Modelo
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train_scaled, y_train)

    # Probabilidades para los nodos sin etiqueta
    probas = clf.predict_proba(X_test_scaled)[:, 1]
    result = pd.Series(probas, index=X_test.index)

    return result

In [13]:
ans_salary_preds = salary_predictions()
assert type(ans_salary_preds) == pd.core.series.Series, "You must return a Pandas series"
assert len(ans_salary_preds) == 252, "The series must be of length 252"


### Part 2B - New Connections Prediction

For the last part of this assignment, you will predict future connections between employees of the network. The future connections information has been loaded into the variable `future_connections`. The index is a tuple indicating a pair of nodes that currently do not have a connection, and the `Future Connection` column indicates if an edge between those two nodes will exist in the future, where a value of 1.0 indicates a future connection.

In [11]:
future_connections = pd.read_csv('assets/Future_Connections.csv', index_col=0, converters={0: eval})
future_connections.head(10)

,Future Connection
"(6, 840)",0.0
"(4, 197)",0.0
"(620, 979)",0.0
"(519, 872)",0.0
"(382, 423)",0.0
"(97, 226)",1.0
"(349, 905)",0.0
"(429, 860)",0.0
"(309, 989)",0.0
"(468, 880)",0.0


Using network `G` and `future_connections`, identify the edges in `future_connections` with missing values and predict whether or not these edges will have a future connection.

To accomplish this, you will need to:      
1. Create a matrix of features of your choice for the edges found in `future_connections` using Networkx     
2. Train a sklearn classifier on those edges in `future_connections` that have `Future Connection` data     
3. Predict a probability of the edge being a future connection for those edges in `future_connections` where `Future Connection` is missing.



Your predictions will need to be given as the probability of the corresponding edge being a future connection.

The evaluation metric for this assignment is the Area Under the ROC Curve (AUC).

Your grade will be based on the AUC score computed for your classifier. A model which with an AUC of 0.75 or higher will recieve full points.

Using your trained classifier, return a series of length 122112 with the data being the probability of the edge being a future connection, and the index being the edge as represented by a tuple of nodes.

    Example:
    
        (107, 348)    0.35
        (542, 751)    0.40
        (20, 426)     0.55
        (50, 989)     0.35
                  ...
        (939, 940)    0.15
        (555, 905)    0.35
        (75, 101)     0.65
        Length: 122112, dtype: float64

In [14]:
def new_connections_predictions():
    import pickle
    import pandas as pd
    import networkx as nx
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.preprocessing import StandardScaler

    # Cargar el grafo y el dataframe de conexiones futuras
    G = pickle.load(open('assets/email_prediction_NEW.txt', 'rb'))
    future_connections = pd.read_csv('assets/Future_Connections.csv', 
                                     index_col=0, 
                                     converters={0: eval})

    # Función para extraer features para cada par de nodos
    def get_features(df, G):
        df_feat = df.copy()
        # Common neighbors
        df_feat['common_neighbors'] = df_feat.index.map(lambda x: len(list(nx.common_neighbors(G, x[0], x[1]))))
        # Preferential attachment
        pa = {(u, v): pa_val for u, v, pa_val in nx.preferential_attachment(G, df_feat.index)}
        df_feat['preferential_attachment'] = df_feat.index.map(lambda x: pa.get((x[0], x[1]), pa.get((x[1], x[0]), 0)))
        # Jaccard coefficient
        jc = {(u, v): jc_val for u, v, jc_val in nx.jaccard_coefficient(G, df_feat.index)}
        df_feat['jaccard'] = df_feat.index.map(lambda x: jc.get((x[0], x[1]), jc.get((x[1], x[0]), 0)))
        # Resource allocation index
        ra = {(u, v): ra_val for u, v, ra_val in nx.resource_allocation_index(G, df_feat.index)}
        df_feat['resource_allocation'] = df_feat.index.map(lambda x: ra.get((x[0], x[1]), ra.get((x[1], x[0]), 0)))
        # Adamic/Adar index
        aa = {(u, v): aa_val for u, v, aa_val in nx.adamic_adar_index(G, df_feat.index)}
        df_feat['adamic_adar'] = df_feat.index.map(lambda x: aa.get((x[0], x[1]), aa.get((x[1], x[0]), 0)))
        # Preferential attachment (degree product)
        df_feat['degree_product'] = df_feat.index.map(lambda x: G.degree(x[0]) * G.degree(x[1]))
        return df_feat

    # Features para los pares con etiqueta (train) y sin etiqueta (test)
    train_df = future_connections[~future_connections['Future Connection'].isnull()]
    test_df = future_connections[future_connections['Future Connection'].isnull()]

    X_train = get_features(train_df, G).drop('Future Connection', axis=1)
    y_train = train_df['Future Connection']
    X_test = get_features(test_df, G).drop('Future Connection', axis=1)

    # Escalar features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Modelo
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train_scaled, y_train)

    # Predecir probabilidades para los pares test
    y_proba = clf.predict_proba(X_test_scaled)[:, 1]
    prob_series = pd.Series(y_proba, index=X_test.index)

    return prob_series

In [15]:
ans_prob_preds = new_connections_predictions()
assert type(ans_prob_preds) == pd.core.series.Series, "You must return a Pandas series"
assert len(ans_prob_preds) == 122112, "The series must be of length 122112"
